# You Only Look Once: Unified, Real-Time Object Detection (YOLO)

**Paper:** [arXiv:1506.02640](https://arxiv.org/abs/1506.02640)  
**Authors:** Joseph Redmon, Santosh Divvala, Ross Girshick, Ali Farhadi (2015)

This notebook implements a simplified YOLO object detector from scratch in PyTorch.  
YOLO reframes object detection as a single regression problem: one neural network pass  
predicts all bounding boxes and class probabilities simultaneously.

**Simplifications vs the paper:**
- Synthetic dataset (colored shapes) instead of PASCAL VOC
- B=1 box per cell (paper uses B=2)
- 3 classes instead of 20
- 6 conv layers instead of 24
- 224x224 input instead of 448x448
- ~25 epochs instead of 120

## 1. Imports & Configuration

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image, ImageDraw
import random
import math
import os

# --- Hyperparameters ---
S = 7          # Grid size (7x7)
B = 1          # Boxes per cell (simplified)
C = 3          # Number of classes (circle, rectangle, triangle)
IMG_SIZE = 224 # Input image size
LAMBDA_COORD = 4.0    # Weight for localization loss
LAMBDA_NOOBJ = 0.5    # Weight for no-object confidence loss
BATCH_SIZE = 32
EPOCHS = 3
LR = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


## 2. Synthetic Dataset

We generate images with randomly placed colored shapes (circles, rectangles, triangles)  
on plain backgrounds. Each image contains 1-3 objects with bounding box annotations.

**Target encoding per grid cell (8 values):**  
`[confidence, x_offset, y_offset, w, h, class_0, class_1, class_2]`

In [2]:
CLASS_NAMES = ['circle', 'rectangle', 'triangle']
CLASS_COLORS = [(255, 50, 50), (50, 255, 50), (50, 50, 255)]

class ShapeDataset(Dataset):
    def __init__(self, num_samples=500, img_size=224, S=7, B=1, C=3):
        self.num_samples = num_samples
        self.img_size = img_size
        self.S = S
        self.B = B
        self.C = C
        self.cell_size = img_size / S

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Create blank image
        img = Image.new('RGB', (self.img_size, self.img_size), (0, 0, 0))
        draw = ImageDraw.Draw(img)

        num_objects = random.randint(1, 3)
        boxes = []   # (x1, y1, x2, y2, class_id)
        for _ in range(num_objects):
            cls_id = random.randint(0, self.C - 1)
            color = CLASS_COLORS[cls_id]
            # Random size
            w = random.randint(30, 80)
            h = random.randint(30, 80)
            cx = random.randint(w // 2 + 5, self.img_size - w // 2 - 5)
            cy = random.randint(h // 2 + 5, self.img_size - h // 2 - 5)
            x1, y1 = cx - w // 2, cy - h // 2
            x2, y2 = cx + w // 2, cy + h // 2

            if cls_id == 0:    # circle
                draw.ellipse([x1, y1, x2, y2], fill=color)
            elif cls_id == 1:  # rectangle
                draw.rectangle([x1, y1, x2, y2], fill=color)
            else:              # triangle
                draw.polygon([(cx, y1), (x1, y2), (x2, y2)], fill=color)

            boxes.append((x1, y1, x2, y2, cls_id))

        # Convert image to tensor
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_tensor = torch.from_numpy(img_array).permute(2, 0, 1)  # HWC -> CHW

        # Build target tensor: (S, S, 5*B + C)
        target = torch.zeros(self.S, self.S, 5 * self.B + self.C)
        for (x1, y1, x2, y2, cls_id) in boxes:
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            w, h = x2 - x1, y2 - y1
            grid_x = int(cx / self.cell_size)
            grid_y = int(cy / self.cell_size)
            grid_x = min(grid_x, self.S - 1)
            grid_y = min(grid_y, self.S - 1)

            # Normalized coordinates within cell
            x_offset = (cx / self.cell_size) - grid_x
            y_offset = (cy / self.cell_size) - grid_y
            w_norm = w / self.img_size
            h_norm = h / self.img_size

            target[grid_y, grid_x, 0] = 1.0          # confidence
            target[grid_y, grid_x, 1] = x_offset
            target[grid_y, grid_x, 2] = y_offset
            target[grid_y, grid_x, 3] = w_norm
            target[grid_y, grid_x, 4] = h_norm
            target[grid_y, grid_x, 5 + cls_id] = 1.0  # class one-hot

        return img_tensor, target

# Create datasets
train_ds = ShapeDataset(num_samples=200, img_size=IMG_SIZE, S=S, B=B, C=C)
test_ds = ShapeDataset(num_samples=50, img_size=IMG_SIZE, S=S, B=B, C=C)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train: {len(train_ds)} samples, Test: {len(test_ds)} samples")

Train: 200 samples, Test: 50 samples


## 3. YOLO Network Architecture

Simplified version of the paper's 24-conv-layer network.  
We use 8 conv layers (with 1x1 reductions + 3x3 convs, inspired by GoogLeNet/NIN)  
followed by 2 fully connected layers. The output is reshaped to (S, S, 5*B + C).

In [3]:
class YOLONet(nn.Module):
    def __init__(self, S=7, B=1, C=3):
        super().__init__()
        self.S = S
        self.B = B
        self.C = C
        self.output_dim = 5 * B + C  # 8

        # --- Conv backbone ---
        self.conv1 = nn.Conv2d(3, 16, 3, stride=1, padding=1)    # 224 -> 224
        self.conv2 = nn.Conv2d(16, 32, 3, stride=2, padding=1)   # 224 -> 112
        self.conv3 = nn.Conv2d(32, 64, 1)                         # 112 -> 112
        self.conv4 = nn.Conv2d(64, 128, 3, stride=2, padding=1)  # 112 -> 56
        self.conv5 = nn.Conv2d(128, 256, 1)                       # 56 -> 56
        self.conv6 = nn.Conv2d(256, 512, 3, stride=2, padding=1) # 56 -> 28
        self.conv7 = nn.Conv2d(512, 512, 3, stride=2, padding=1) # 28 -> 14
        self.conv8 = nn.Conv2d(512, 1024, 3, stride=2, padding=1)# 14 -> 7

        self.leaky = nn.LeakyReLU(0.1)

        # --- Detection head ---
        flat_dim = 1024 * S * S
        self.fc1 = nn.Linear(flat_dim, 1024)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(1024, S * S * self.output_dim)

    def forward(self, x):
        x = self.leaky(self.conv1(x))   # (B, 16, 224, 224)
        x = self.leaky(self.conv2(x))   # (B, 32, 112, 112)
        x = self.leaky(self.conv3(x))   # (B, 64, 112, 112)
        x = self.leaky(self.conv4(x))   # (B, 128, 56, 56)
        x = self.leaky(self.conv5(x))   # (B, 256, 56, 56)
        x = self.leaky(self.conv6(x))   # (B, 512, 28, 28)
        x = self.leaky(self.conv7(x))   # (B, 512, 14, 14)
        x = self.leaky(self.conv8(x))   # (B, 1024, 7, 7)

        x = x.flatten(1)                # (B, 1024*7*7)
        x = self.leaky(self.fc1(x))     # (B, 1024)
        x = self.dropout(x)
        x = torch.sigmoid(self.fc2(x))  # (B, S*S*output_dim)
        x = x.view(-1, self.S, self.S, self.output_dim)  # (B, 7, 7, 8)
        return x

model = YOLONet(S=S, B=B, C=C).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 60,156,712


## 4. YOLO Loss Function

Implements the multi-part sum-squared-error loss from the paper (Equation 2):

1. **Coordinate loss** (lambda_coord=4): penalizes x, y, sqrt(w), sqrt(h) errors at cells containing objects  
2. **Object confidence loss**: penalizes confidence predictions at cells with objects  
3. **No-object confidence loss** (lambda_noobj=0.5): penalizes non-zero confidence at empty cells  
4. **Classification loss**: penalizes class probability errors at cells with objects

In [4]:
class YOLOLoss(nn.Module):
    def __init__(self, S=7, B=1, C=3, lambda_coord=4.0, lambda_noobj=0.5):
        super().__init__()
        self.S = S
        self.B = B
        self.C = C
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
        self.mse = nn.MSELoss(reduction='sum')

    def forward(self, pred, target):
        # pred:   (batch, S, S, 5*B + C)
        # target: (batch, S, S, 5*B + C)
        batch_size = pred.size(0)

        # Masks: which cells contain objects?
        obj_mask = target[..., 0] > 0.5    # (batch, S, S)
        noobj_mask = ~obj_mask              # (batch, S, S)

        # --- Coordinate loss ---
        coord_loss = 0.0
        if obj_mask.any():
            # x, y offsets
            pred_xy = pred[..., 1:3][obj_mask]       # (N_obj, 2)
            true_xy = target[..., 1:3][obj_mask]     # (N_obj, 2)
            coord_loss = coord_loss + self.mse(pred_xy, true_xy)

            # w, h with square-root parameterization
            pred_wh = pred[..., 3:5][obj_mask]       # (N_obj, 2)
            true_wh = target[..., 3:5][obj_mask]     # (N_obj, 2)
            pred_wh_sqrt = torch.sqrt(pred_wh.clamp(min=1e-8))
            true_wh_sqrt = torch.sqrt(true_wh.clamp(min=1e-8))
            coord_loss = coord_loss + self.mse(pred_wh_sqrt, true_wh_sqrt)

        # --- Confidence loss (object cells) ---
        obj_conf_loss = torch.tensor(0.0, device=pred.device)
        if obj_mask.any():
            pred_conf_obj = pred[..., 0][obj_mask]
            true_conf_obj = target[..., 0][obj_mask]
            obj_conf_loss = self.mse(pred_conf_obj, true_conf_obj)

        # --- Confidence loss (no-object cells) ---
        noobj_conf_loss = torch.tensor(0.0, device=pred.device)
        if noobj_mask.any():
            pred_conf_noobj = pred[..., 0][noobj_mask]
            true_conf_noobj = target[..., 0][noobj_mask]
            noobj_conf_loss = self.mse(pred_conf_noobj, true_conf_noobj)

        # --- Classification loss ---
        cls_loss = torch.tensor(0.0, device=pred.device)
        if obj_mask.any():
            pred_cls = pred[..., 5:][obj_mask]       # (N_obj, C)
            true_cls = target[..., 5:][obj_mask]     # (N_obj, C)
            cls_loss = self.mse(pred_cls, true_cls)

        # --- Total ---
        total = (
            self.lambda_coord * coord_loss
            + obj_conf_loss
            + self.lambda_noobj * noobj_conf_loss
            + cls_loss
        ) / batch_size

        return total, coord_loss.item() if isinstance(coord_loss, torch.Tensor) else coord_loss, \
               obj_conf_loss.item() if isinstance(obj_conf_loss, torch.Tensor) else obj_conf_loss, \
               cls_loss.item() if isinstance(cls_loss, torch.Tensor) else cls_loss

criterion = YOLOLoss(S=S, B=B, C=C, lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ)

## 5. Intersection-over-Union (IoU)

Standard IoU metric for evaluating bounding box overlap.

In [5]:
def compute_iou(box1, box2):
    # Compute IoU between two boxes in (x1, y1, x2, y2) format.
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])

    inter_area = max(0, x2_inter - x1_inter) * max(0, y2_inter - y1_inter)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area

    if union_area == 0:
        return 0.0
    return inter_area / union_area

def non_max_suppression(boxes, scores, iou_threshold=0.4):
    # Remove overlapping boxes with lower confidence.
    if len(boxes) == 0:
        return []
    indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    keep = []
    while indices:
        i = indices.pop(0)
        keep.append(i)
        indices = [j for j in indices if compute_iou(boxes[i], boxes[j]) < iou_threshold]
    return keep

## 6. Decoding Predictions

Convert the network's (S, S, 8) output tensor into bounding boxes + class labels.

In [6]:
def decode_prediction(pred, S=7, B=1, C=3, conf_thresh=0.5, img_size=224):
    # pred: (S, S, 5*B + C) numpy array
    # Returns: list of (x1, y1, x2, y2, class_id, confidence)
    cell_size = img_size / S
    boxes = []
    scores = []
    class_ids = []

    for gy in range(S):
        for gx in range(S):
            conf = pred[gy, gx, 0]
            if conf < conf_thresh:
                continue
            x_offset = pred[gy, gx, 1]
            y_offset = pred[gy, gx, 2]
            w = pred[gy, gx, 3]
            h = pred[gy, gx, 4]

            cx = (gx + x_offset) * cell_size
            cy = (gy + y_offset) * cell_size
            bw = w * img_size
            bh = h * img_size
            x1 = cx - bw / 2
            y1 = cy - bh / 2
            x2 = cx + bw / 2
            y2 = cy + bh / 2

            # Class prediction
            class_probs = pred[gy, gx, 5:5 + C]
            cls_id = int(np.argmax(class_probs))
            score = conf * class_probs[cls_id]

            boxes.append((x1, y1, x2, y2))
            scores.append(float(score))
            class_ids.append(cls_id)

    # NMS
    keep_idx = non_max_suppression(boxes, scores, iou_threshold=0.4)
    results = [(boxes[i][0], boxes[i][1], boxes[i][2], boxes[i][3], class_ids[i], scores[i]) for i in keep_idx]
    return results

## 7. Training Loop

In [7]:
optimizer = optim.Adam(model.parameters(), lr=LR)
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    coord_l_val = 0.0
    conf_l_val = 0.0
    cls_l_val = 0.0

    for imgs, targets in train_loader:
        imgs = imgs.to(DEVICE)
        targets = targets.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss, coord_l, conf_l, cls_l = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1
        coord_l_val = coord_l
        conf_l_val = conf_l
        cls_l_val = cls_l

    avg_loss = epoch_loss / max(n_batches, 1)
    train_losses.append(avg_loss)

    if (epoch + 1) % 1 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}]  Loss: {avg_loss:.4f}  (coord={coord_l_val:.1f}, conf={conf_l_val:.1f}, cls={cls_l_val:.1f})")

print("Training complete!")

Epoch [1/3]  Loss: 14.3339  (coord=5.8, conf=2.2, cls=16.3)


Epoch [2/3]  Loss: 14.4130  (coord=13.7, conf=12.0, cls=17.2)


Epoch [3/3]  Loss: 12.9693  (coord=15.7, conf=14.0, cls=10.0)
Training complete!


## 8. Training Loss Curve

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(train_losses, 'b-', linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('YOLO Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('yolo_training_loss.png', dpi=100)
plt.show()
print("Loss plot saved.")

Loss plot saved.


## 9. Detection Visualization

Test the trained model on sample images and visualize predicted bounding boxes.

In [9]:
def draw_detections(img_tensor, detections, class_names, class_colors):
    # Draw bounding boxes on an image tensor.
    img = img_tensor.permute(1, 2, 0).numpy()
    img = (img * 255).astype(np.uint8).copy()
    pil_img = Image.fromarray(img)
    draw = ImageDraw.Draw(pil_img)

    for (x1, y1, x2, y2, cls_id, score) in detections:
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        color = class_colors[cls_id]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        label = f"{class_names[cls_id]}: {score:.2f}"
        draw.text((x1, max(0, y1 - 12)), label, fill=color)

    return pil_img

# Run inference on test samples
model.eval()
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

with torch.no_grad():
    for i in range(min(8, len(test_ds))):
        img, target = test_ds[i]
        img_input = img.unsqueeze(0).to(DEVICE)
        pred = model(img_input)
        pred_np = pred[0].cpu().numpy()
        dets = decode_prediction(pred_np, S=S, B=B, C=C, conf_thresh=0.3, img_size=IMG_SIZE)

        result_img = draw_detections(img, dets, CLASS_NAMES, CLASS_COLORS)
        axes[i].imshow(result_img)
        axes[i].axis('off')
        n_dets = len(dets)
        axes[i].set_title(f'{n_dets} detections')

plt.suptitle('YOLO Detection Results on Synthetic Shapes', fontsize=14)
plt.tight_layout()
plt.savefig('yolo_detections.png', dpi=100)
plt.show()
print(f"Detection visualization complete. Showed {min(8, len(test_ds))} test images.")

Detection visualization complete. Showed 8 test images.


## 10. Evaluation Metrics

In [10]:
def evaluate_model(model, dataset, S=7, B=1, C=3, conf_thresh=0.3, iou_thresh=0.5):
    # Compute mean IoU and detection accuracy on the dataset.
    model.eval()
    total_iou = 0.0
    n_correct = 0
    n_total = 0

    with torch.no_grad():
        for i in range(min(50, len(dataset))):
            img, target = dataset[i]
            img_input = img.unsqueeze(0).to(DEVICE)
            pred = model(img_input)
            pred_np = pred[0].cpu().numpy()
            dets = decode_prediction(pred_np, S=S, B=B, C=C, conf_thresh=conf_thresh, img_size=IMG_SIZE)

            # Get ground truth boxes
            gt_boxes = []
            for gy in range(S):
                for gx in range(S):
                    if target[gy, gx, 0] > 0.5:
                        x_off = target[gy, gx, 1]
                        y_off = target[gy, gx, 2]
                        w = target[gy, gx, 3]
                        h = target[gy, gx, 4]
                        cell_size = IMG_SIZE / S
                        cx = (gx + x_off) * cell_size
                        cy = (gy + y_off) * cell_size
                        bw = w * IMG_SIZE
                        bh = h * IMG_SIZE
                        gt_boxes.append((cx - bw/2, cy - bh/2, cx + bw/2, cy + bh/2))

            # Match detections to ground truth
            for gt_box in gt_boxes:
                best_iou = 0
                for det in dets:
                    det_box = (det[0], det[1], det[2], det[3])
                    iou = compute_iou(gt_box, det_box)
                    best_iou = max(best_iou, iou)
                total_iou += best_iou
                if best_iou > iou_thresh:
                    n_correct += 1
                n_total += 1

    mean_iou = total_iou / max(n_total, 1)
    accuracy = n_correct / max(n_total, 1)
    return mean_iou, accuracy

mean_iou, det_acc = evaluate_model(model, test_ds, S=S, B=B, C=C)
print(f"Mean IoU: {mean_iou:.4f}")
print(f"Detection Accuracy (IoU > 0.5): {det_acc:.4f}")
print(f"Total parameters: {total_params:,}")
print(f"Final training loss: {train_losses[-1]:.4f}")

Mean IoU: 0.0000
Detection Accuracy (IoU > 0.5): 0.0000
Total parameters: 60,156,712
Final training loss: 12.9693


## Summary

This notebook implemented a simplified YOLO (You Only Look Once) object detector from scratch:

- **Grid-based detection:** 7x7 grid with each cell predicting 1 bounding box + 3 class probabilities
- **Single forward pass:** One network evaluation produces all detections
- **Multi-part loss:** Coordinate (lambda=4), confidence, no-object (lambda=0.5), and classification components
- **Square-root parameterization:** For bounding box width/height to weight small-box errors more
- **Non-maximal suppression:** Post-processing to remove duplicate detections

**Key takeaway from the paper:** YOLO's innovation was not accuracy (it traded some mAP for speed)  
but the *unified* framing -- treating detection as a single regression problem optimizable end-to-end.  
This paradigm shift enabled real-time detection and spawned the entire YOLO family (v2 through v11).

In [ ]:
# --- Auto-injected by kaggle_validator: save executed notebook with outputs ---

import shutil, os

# Kaggle internally stores the executed notebook as __notebook__.ipynb

if os.path.exists('/kaggle/working/__notebook__.ipynb'):

    shutil.copy('/kaggle/working/__notebook__.ipynb', '/kaggle/working/executed_solution.ipynb')

    print('Executed notebook saved to output folder.')

else:

    # Fallback: try alternative path

    import glob

    candidates = glob.glob('/kaggle/working/*.ipynb')

    print(f'Available notebooks in output: {candidates}')
